# Imports

In [21]:
import warnings

import pandas as pd

from functions.loading import load_data

from functions.training_pipeline import training_pipeline
from functions.models import xgboost_model, catboost_model, lgbm_model

warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

# Parameters definition

In [22]:
path_rawdata = 'data/raw_data/'
restricted_features=False
save=False,

if restricted_features == False:
    path_intermediary = 'data/intermediary_data/unrestricted_features/'
    path_models = 'models/unrestricted_features/'
    path_results = 'results/unrestricted_features/'
    path_plot = path_results +'plot/'

elif restricted_features == True:
    path_intermediary = 'data/intermediary_data/restricted_features/'
    path_models = 'models/restricted_features/'
    path_results = 'results/restricted_features/'
    path_plot = path_results +'plot/'

In [7]:
extract_dataset = pd.read_parquet(path_rawdata + 'dataset_raw.parquet')

In [8]:
raw_dataset_filtered = extract_dataset[(extract_dataset.Revenue.notnull()) & (extract_dataset.GICSName.notnull())]

In [9]:
raw_dataset_filtered = raw_dataset_filtered[['FinalEikonID', 'Name',  'Ticker', 'LEI',
        'CountryHQ',  'GICSSector',
       'GICSGroup', 'GICSInd', 'GICSSubInd', 'GICSName', 'FiscalYear', 'CF1',
       'CF2', 'CF3', 'EBITDA', 'EBIT', 'CapEx', 'GPPE',
       'NPPE', 'AccuDep', 'INTAN', 'COGS', 'GMAR', 'Asset', 'LTDebt', 'EMP',
       'ENEProduce', 'ENEConsume', 'Revenue', 'CF123']]



In [12]:
raw_dataset_filtered.columns = ['company_id', 'company_name', 'ticker', 'lei', 'country_hq', 'gics_sector', 'gics_group', 'gics_ind', 'gics_sub_ind', 'gics_name', 'fiscal_year', 'cf1', 'cf2', 'cf3', 'ebitda', 'ebit', 'capex', 'gppe', 'nppe', 'accu_dep', 'intan', 'cogs', 'gmar', 'asset', 'lt_debt', 'employees', 'energy_produced', 'energy_consumed', 'revenue', 'cf123']
raw_dataset_filtered.to_parquet("data/raw_data/input_dataset_full_extract.parquet")

# Data Loading 

In [15]:
preprocessed_dataset = load_data(
    path_rawdata, filter_outliers=False, threshold_under=1.5, threshold_over=2.5, save=False
)

In [16]:
preprocessed_dataset 

,company_id,company_name,ticker,lei,country_hq,gics_sector,gics_group,gics_ind,gics_sub_ind,gics_name,...,price,status,area,year,fuel_intensity,income_group,region,age,cap_inten,leverage
0,AIAS.CY,Aias Investment Public Ltd,AIAS,2138001G526W1KXZU288,Cyprus,40.0,4020.0,402030.0,40203010.0,Asset Management & Custody Banks,...,NaN,Yes,Cyprus,2018.0,638.34,H,Europe and Central Asia,NaN,NaN,0.0
1,AIAS.CY,Aias Investment Public Ltd,AIAS,2138001G526W1KXZU288,Cyprus,40.0,4020.0,402030.0,40203010.0,Asset Management & Custody Banks,...,NaN,Yes,Cyprus,2017.0,644.00,H,Europe and Central Asia,NaN,NaN,0.0
2,AIAS.CY,Aias Investment Public Ltd,AIAS,2138001G526W1KXZU288,Cyprus,40.0,4020.0,402030.0,40203010.0,Asset Management & Custody Banks,...,NaN,Yes,Cyprus,2016.0,642.13,H,Europe and Central Asia,NaN,NaN,0.0
3,AIAS.CY,Aias Investment Public Ltd,AIAS,2138001G526W1KXZU288,Cyprus,40.0,4020.0,402030.0,40203010.0,Asset Management & Custody Banks,...,NaN,Yes,Cyprus,2015.0,643.17,H,Europe and Central Asia,NaN,NaN,0.0
4,AIAS.CY,Aias Investment Public Ltd,AIAS,2138001G526W1KXZU288,Cyprus,40.0,4020.0,402030.0,40203010.0,Asset Management & Custody Banks,...,NaN,Yes,Cyprus,2014.0,652.07,H,Europe and Central Asia,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734144,PBYI.OQ,Puma Biotechnology Inc,PBYI,<NA>,United States of America,35.0,3520.0,352010.0,35201010.0,Biotechnology,...,NaN,Yes,United States of America,2014.0,467.31,H,North America,5.553429,NaN,0.0
734145,PBYI.OQ,Puma Biotechnology Inc,PBYI,<NA>,United States of America,35.0,3520.0,352010.0,35201010.0,Biotechnology,...,NaN,Yes,United States of America,2013.0,470.62,H,North America,5.633570,NaN,0.0
734146,PBYI.OQ,Puma Biotechnology Inc,PBYI,<NA>,United States of America,35.0,3520.0,352010.0,35201010.0,Biotechnology,...,NaN,Yes,United States of America,2012.0,470.47,H,North America,6.622642,NaN,0.0
734147,PBYI.OQ,Puma Biotechnology Inc,PBYI,<NA>,United States of America,35.0,3520.0,352010.0,35201010.0,Biotechnology,...,NaN,Yes,United States of America,2011.0,484.12,H,North America,62.978182,NaN,0.0


In [6]:
from functions.preprocessing import encoding

In [4]:
preprocessed_dataset = pd.read_parquet(path_rawdata + "cgee_preprocessed_dataset_2023.parquet")

In [7]:
predict_dataset = encoding(preprocessed_dataset, path_intermediary , train = False, restricted_features=False) 

In [8]:
predict_dataset_set = predict_dataset[(predict_dataset.fiscal_year >= 2010) & (predict_dataset.fiscal_year <= 2022)]

In [18]:
predict_dataset_set.reset_index(drop=True)

,company_id,company_name,ticker,lei,country_hq,gics_sector,gics_group,gics_ind,gics_sub_ind,gics_name,...,asset_log,lt_debt_log,cf1_log,cf2_log,cf3_log,cf123_log,energy_consumed_log,energy_produced_log,ebit_log,ebitda_log
0,AIAS.CY,Aias Investment Public Ltd,AIAS,2138001G526W1KXZU288,Cyprus,40.0,4020.0,402030.0,40203010.0,Asset Management & Custody Banks,...,7.250977,7.305645,NaN,NaN,NaN,NaN,4.246336,5.984924,10.888861,10.888861
1,AIAS.CY,Aias Investment Public Ltd,AIAS,2138001G526W1KXZU288,Cyprus,40.0,4020.0,402030.0,40203010.0,Asset Management & Custody Banks,...,7.268699,7.305645,NaN,NaN,NaN,NaN,4.919184,5.834606,10.888858,10.888858
2,AIAS.CY,Aias Investment Public Ltd,AIAS,2138001G526W1KXZU288,Cyprus,40.0,4020.0,402030.0,40203010.0,Asset Management & Custody Banks,...,7.192868,7.305645,NaN,NaN,NaN,NaN,4.246336,5.984924,10.888862,10.888862
3,AIAS.CY,Aias Investment Public Ltd,AIAS,2138001G526W1KXZU288,Cyprus,40.0,4020.0,402030.0,40203010.0,Asset Management & Custody Banks,...,7.219161,7.305645,NaN,NaN,NaN,NaN,4.246336,5.984924,10.888863,10.888863
4,AIAS.CY,Aias Investment Public Ltd,AIAS,2138001G526W1KXZU288,Cyprus,40.0,4020.0,402030.0,40203010.0,Asset Management & Custody Banks,...,7.261210,7.305645,NaN,NaN,NaN,NaN,4.919184,5.834606,10.888852,10.888852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
519847,PBYI.OQ,Puma Biotechnology Inc,PBYI,<NA>,United States of America,35.0,3520.0,352010.0,35201010.0,Biotechnology,...,8.212667,7.305645,NaN,NaN,NaN,NaN,4.572041,NaN,10.888063,10.888066
519848,PBYI.OQ,Puma Biotechnology Inc,PBYI,<NA>,United States of America,35.0,3520.0,352010.0,35201010.0,Biotechnology,...,8.020345,7.305645,NaN,NaN,NaN,NaN,4.572041,NaN,10.888554,10.888556
519849,PBYI.OQ,Puma Biotechnology Inc,PBYI,<NA>,United States of America,35.0,3520.0,352010.0,35201010.0,Biotechnology,...,8.182190,7.305645,NaN,NaN,NaN,NaN,4.572041,NaN,10.888444,10.888445
519850,PBYI.OQ,Puma Biotechnology Inc,PBYI,<NA>,United States of America,35.0,3520.0,352010.0,35201010.0,Biotechnology,...,7.746792,7.305645,NaN,NaN,NaN,NaN,4.572041,NaN,10.888805,10.888805


In [19]:
predict_dataset_set.to_parquet(path_rawdata+"predict_dataset.parquet")

### CGEE sample

In [24]:
input_dataset = pd.read_parquet(path_rawdata +"cgee_preprocessed_dataset_2023.parquet")
data_with_high_revenue2021 = input_dataset[input_dataset.fiscal_year == 2021]
data_with_high_revenue2021 = data_with_high_revenue2021.sort_values(by='revenue', ascending=False)
companies = list(data_with_high_revenue2021.company_name.unique())
companies50 = companies[:50]

In [69]:
estimated_dataset = pd.read_csv(path_results +"estimated_scopes.csv")
estimated_dataset = estimated_dataset.rename(columns={"cf1_log_estimated":"estimated_scope_1_emissions","cf2_log_estimated":"estimated_scope_2_emissions","cf3_log_estimated":"estimated_scope_3_emissions","cf123_log_estimated":"estimated_scope_123_emissions"})
lst_years = [2021,2020,2019,2018]
sample_dataset = estimated_dataset[estimated_dataset.company_name.isin(companies50)]
sample_dataset = sample_dataset[sample_dataset.fiscal_year.isin(lst_years)]
sample_dataset = sample_dataset.reset_index(drop=True).sort_values(
        by=["company_id","fiscal_year"])
sample_dataset.to_excel("sample_dataset.xlsx")



In [68]:
sample_dataset

,company_id,company_name,ticker,lei,fiscal_year,estimated_scope_1_emissions,estimated_scope_2_emissions,estimated_scope_3_emissions,estimated_scope_123_emissions
3,005930.KS,Samsung Electronics Co Ltd,005930,9884007ER46L6N7EI764,2018,1.454597e+07,6.335113e+07,1.310502e+09,6.367912e+08
0,005930.KS,Samsung Electronics Co Ltd,005930,9884007ER46L6N7EI764,2019,1.649345e+07,6.579408e+07,8.932625e+08,5.904341e+08
2,005930.KS,Samsung Electronics Co Ltd,005930,9884007ER46L6N7EI764,2020,1.750255e+07,5.771817e+07,9.598329e+08,6.880221e+08
1,005930.KS,Samsung Electronics Co Ltd,005930,9884007ER46L6N7EI764,2021,2.396200e+07,6.058968e+07,1.271855e+09,1.127873e+09
7,2222.SE,Saudi Arabian Oil Co,2222,5586006WD91QHB7J4X50,2018,4.244844e+08,5.784359e+07,7.031932e+09,6.009589e+09
...,...,...,...,...,...,...,...,...,...
192,WMT.N,Walmart Inc,WMT,Y87794H0US1R65VBXU25,2021,1.416452e+07,3.318405e+07,1.666425e+07,1.888669e+08
196,XOM.N,Exxon Mobil Corp,XOM,J3WHBG0MTS7O8ZVMDC91,2018,1.198669e+09,5.325665e+07,5.668992e+09,8.792277e+09
197,XOM.N,Exxon Mobil Corp,XOM,J3WHBG0MTS7O8ZVMDC91,2019,1.216330e+09,4.880624e+07,6.268604e+09,8.542865e+09
198,XOM.N,Exxon Mobil Corp,XOM,J3WHBG0MTS7O8ZVMDC91,2020,9.005578e+08,3.245356e+07,4.895214e+09,5.185755e+09


In [72]:
estimated_dataset["company_id_year"] = estimated_dataset["company_id"] +"_" + estimated_dataset["fiscal_year"].astype(str)
lst_columns_gcp = ["company_id_year","company_id","company_name","ticker","lei","fiscal_year",	"estimated_scope_1_emissions",	"estimated_scope_2_emissions",	"estimated_scope_3_emissions",	"estimated_scope_123_emissions"]
estimated_dataset_final = estimated_dataset[lst_columns_gcp].reset_index(drop=True)
estimated_dataset_final.to_csv("estimated_dataset_final.csv", index=False)

In [73]:
estimated_dataset_final.to_parquet("estimated_dataset_final.parquet", index=False)